In [1]:
import pandas as pd
from konlpy.tag import Mecab
from gensim import corpora
from gensim.models.ldamodel import LdaModel
import networkx as nx
import numpy as np
import tqdm

In [6]:
RESNET_DATA_PATH = '../data/interim/resnet90deg.csv'
SENTIMENTAL_SCORE_PAHT = '../data/interim/sendimental_score.csv'

In [7]:
df = pd.read_csv(RESNET_DATA_PATH)
sentimental = pd.read_csv(SENTIMENTAL_SCORE_PAHT)

In [8]:
df.head(1)

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자,alpha,경도,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,중앙하이츠,59.91,10.0,1998.0,덕릉로84길 7,6.766156,22.0,2020-07-11,0.266667,127.076754,...,2.004844,-2.512005,-1.100615,-0.027693,-1.204624,2.213226,1.975165,0.90034,0.369453,0.569277


# 뉴스 데이터 준비 및 전처리

In [9]:
DEEP_PATH = '../data/interim/news/deep_search_news.csv'
STOPWORD_PATH = '../data/raw/news/stopwords-ko.txt'

POSITIVE_PATH = '../data/raw/news/sentiment/positive.txt'
NEGATIVE_PATH = '../data/raw/news/sentiment/negative.txt'
NATURAL_PATH = '../data/raw/news/sentiment/natural.txt'
# 뉴스 데이터 읽기
df = pd.read_csv(DEEP_PATH)
#df = data.head(20) # 테스트용으로 상위 20개만 해봄

# 각종 TXT 파일 불러오기
def load_txt(PATH):
    with open(PATH, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]
# 불용어 불러오기
stopwords = load_txt(STOPWORD_PATH)
# 긍정, 부정, 중립 단어 불러오기
positive = load_txt(POSITIVE_PATH)
negative = load_txt(NEGATIVE_PATH)
natural = load_txt(NATURAL_PATH)

print(f"불용어 단어 예시 : {stopwords[:5]}...")
print(f"긍정 단어 예시 : {positive[:5]}...")
print(f"부정 단어 예시 : {negative[:5]}...")
print(f"중립 단어 예시 : {natural[:5]}...")

불용어 단어 예시 : ['가', '가까스로', '가령', '각', '각각']...
긍정 단어 예시 : ['활황', '급매물', '소진', '강세', '매수세']...
부정 단어 예시 : ['침체', '급매', '투매', '하락', '폭락']...
중립 단어 예시 : ['부동산', '아파트', '주택', '토지', '건물']...


In [10]:
df['content']

0        금융위원회는 이날 문자 공지를 통해 "은 위원장이 보유한 세종시 아파트에 대한 매수...
1        은성수 금융위원장이 8일 세종시에서 보유한 아파트의 매매 합의를 함에 따라 2주택자...
2        주택대출 규제의 주무 장관인 은성수 금융위원장이 8일 세종시에 보유한 아파트를 매도...
3        8일 금융권에 따르면 은 위원장은 8일 세종시 아파트 매매를 합의하고 가계약금을 수...
4        은 위원장은 지난해 12·16 부동산 대책 발표 후 ‘고위공직자 1주택 보유’ 기조...
                               ...                        
26739    서학개미 열풍 속에 올해 상반기 주요 증권사가 해외주식 거래 수수료로 벌어들인 수익...
26740    부동산원, 전국 주간 아파트 가격 동향 6·27 부동산 대책 발표 이후 서울의 아파...
26741    올해 3월 140.3㎡ 55억원에 매도 앞서 2017년 24.4억에 분양받아 축구선...
26742    보통주 총 1307만 5691주에 매각 가격은 주당 10만 7100원으로 총 1조 ...
26743    ‘청주 센텀 푸르지오 자이’ 8일(월) 무순위 청약 접수 약 1.4만여 가구 규모 ...
Name: content, Length: 26744, dtype: object

In [11]:
# 명사 추출 + 불용어 제거 함수
mecab = Mecab()
def tokenize(text):
    if not isinstance(text, str):
        return []
    return [word for word in mecab.nouns(text) 
            if len(word) > 1 and word not in stopwords]

# 토큰화 + 불용어 제거 적용
df['tokens'] = df['content'].apply(tokenize)

print(f"토큰 예시 : {df['tokens'][0]}")

토큰 예시 : ['금융', '위원회', '이날', '문자', '공지', '위원장', '보유', '종시', '아파트', '매수', '매매', '합의', '계약금', '수령', '당초', '위원장', '잠원동', '아파트', '도담동', '아파트', '본인', '명의', '종시', '아파트', '정세균', '주택', '보유', '권고', '처분', '호가', '수준', '매각', '위원장', '종시', '아파트', '최초', '매도', '호가', '수준', '계약', '성사']


# 2단계 : 토픽 모델링 및 텍스트 랭크

In [12]:
# 토픽 모델링 (LDA)
dictionary = corpora.Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in df['tokens']]
lda_model = LdaModel(corpus, num_topics=8, id2word=dictionary, passes=15)

topics = lda_model.print_topics(num_words=30)
print(f"토픽 모델링 결과 예시 :\n {topics[0]}")

토픽 모델링 결과 예시 :
 (0, '0.054*"사업" + 0.028*"구역" + 0.027*"개발" + 0.026*"재건축" + 0.023*"아파트" + 0.021*"정비" + 0.019*"서울시" + 0.018*"계획" + 0.016*"도시" + 0.016*"토지" + 0.015*"지구" + 0.015*"건축" + 0.014*"조합" + 0.014*"가구" + 0.014*"추진" + 0.012*"지정" + 0.011*"허가" + 0.010*"지역" + 0.010*"선정" + 0.009*"시공사" + 0.008*"공공" + 0.008*"분양" + 0.007*"공사비" + 0.007*"건설" + 0.007*"주택" + 0.006*"입찰" + 0.005*"규모" + 0.005*"민간" + 0.005*"공사" + 0.005*"통과"')


In [13]:
# 텍스트랭크
def text_rank_keywords(tokens):
    g = nx.Graph()
    for i in range(len(tokens) - 1):
        g.add_edge(tokens[i], tokens[i+1])
    pr = nx.pagerank(g, weight='weight')
    return sorted(pr, key=pr.get, reverse=True) # 상위 몇개를 포함할건가?는 논문에 없다

In [14]:
df['textrank_keywords'] = df['tokens'].apply(text_rank_keywords)
print("\n텍스트랭크 키워드:")
print(df[['content', 'textrank_keywords']].head())


텍스트랭크 키워드:
                                             content  \
0  금융위원회는 이날 문자 공지를 통해 "은 위원장이 보유한 세종시 아파트에 대한 매수...   
1  은성수 금융위원장이 8일 세종시에서 보유한 아파트의 매매 합의를 함에 따라 2주택자...   
2  주택대출 규제의 주무 장관인 은성수 금융위원장이 8일 세종시에 보유한 아파트를 매도...   
3  8일 금융권에 따르면 은 위원장은 8일 세종시 아파트 매매를 합의하고 가계약금을 수...   
4  은 위원장은 지난해 12·16 부동산 대책 발표 후 ‘고위공직자 1주택 보유’ 기조...   

                                   textrank_keywords  
0  [아파트, 위원장, 보유, 종시, 수준, 호가, 위원회, 이날, 계약, 문자, 합의...  
1  [위원장, 아파트, 금융, 종시, 매매, 합의, 공지, 저녁, 문자, 오늘, 최근,...  
2  [아파트, 위원장, 종시, 주택, 매매, 금융, 부동산, 보유, 주무, 장관, 규제...  
3      [위원장, 아파트, 종시, 합의, 계약금, 매매, 수령, 지난해, 잠원동, 금융]  
4  [아파트, 위원장, 종시, 고위, 공직자, 발표, 주택, 대책, 보유, 부동산, 전...  


# 3단계 : 감성 사전 기반 감성 점수 산출

In [15]:
# --- 3단계: 감성 사전 기반 감성 점수 산출 ---
print("\n3. 감성 사전 기반 감성 점수 산출 시작...")

def get_sentiment_score(tokens):
    pos_score = sum(1 for word in tokens if word in positive)
    neg_score = sum(1 for word in tokens if word in negative)
    nat_score = sum(1 for word in tokens if word in natural)
    total_words = len(tokens)
    if total_words == 0:
        return 0
    return (pos_score - neg_score) / total_words

df['sentiment_score'] = df['tokens'].apply(get_sentiment_score)

print("\n감성 사전 기반 감성 점수:")
print(df[['content', 'sentiment_score']].head())


3. 감성 사전 기반 감성 점수 산출 시작...

감성 사전 기반 감성 점수:
                                             content  sentiment_score
0  금융위원회는 이날 문자 공지를 통해 "은 위원장이 보유한 세종시 아파트에 대한 매수...        -0.024390
1  은성수 금융위원장이 8일 세종시에서 보유한 아파트의 매매 합의를 함에 따라 2주택자...         0.030303
2  주택대출 규제의 주무 장관인 은성수 금융위원장이 8일 세종시에 보유한 아파트를 매도...        -0.093023
3  8일 금융권에 따르면 은 위원장은 8일 세종시 아파트 매매를 합의하고 가계약금을 수...         0.058824
4  은 위원장은 지난해 12·16 부동산 대책 발표 후 ‘고위공직자 1주택 보유’ 기조...        -0.037037


# 4단계 : 월별 감성 지수 산출 및 예측 모델 통합

In [16]:
# --- 4단계: 월별 감성 지수 산출 및 예측 모델 통합 ---
print("\n4. 월별 감성 지수 산출 및 예측 모델 통합...")


4. 월별 감성 지수 산출 및 예측 모델 통합...


In [17]:
# datetime 변환
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
df['month'] = df['date'].dt.to_period('M')

In [18]:
monthly_sentiment = df.groupby('month')['sentiment_score'].mean().reset_index()
monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

# monthly_sentiment month 컬럼도 period[M]로 변환
monthly_sentiment['month'] = pd.to_datetime(monthly_sentiment['month']).dt.to_period('M')


In [19]:
print("\n월별 감성 지수:")
print(monthly_sentiment)



월별 감성 지수:
      month  sentiment_score
0   2020-07         0.014421
1   2020-08         0.008881
2   2020-09         0.020459
3   2020-10         0.025977
4   2020-11         0.017741
..      ...              ...
58  2025-05         0.033125
59  2025-06         0.030115
60  2025-07         0.009352
61  2025-08         0.025129
62  2025-09         0.026038

[63 rows x 2 columns]


In [20]:
SALE_PATH = '../data/interim/resnet90deg.csv'
sale = pd.read_csv(SALE_PATH)

In [21]:
# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.to_period('M')
drop_col = ['단지명','도로명','계약일자','경도','위도']

In [22]:
sale.columns

Index(['단지명', '전용면적(㎡)', '층', '건축년도', '도로명', '면적당 단가(만원)', '아파트 나이', '계약일자',
       'alpha', '경도',
       ...
       'feature_2039', 'feature_2040', 'feature_2041', 'feature_2042',
       'feature_2043', 'feature_2044', 'feature_2045', 'feature_2046',
       'feature_2047', 'month'],
      dtype='object', length=2060)

In [23]:
sale.drop(drop_col, axis=1, inplace=True)

In [24]:
sale.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,month
0,59.91,10.0,1998.0,6.766156,22.0,0.266667,1.840863,0.455314,1.392201,-1.213092,...,-2.512005,-1.100615,-0.027693,-1.204624,2.213226,1.975165,0.900340,0.369453,0.569277,2020-07
1,59.77,7.0,1996.0,7.499383,24.0,1.000000,0.512528,0.327763,0.756215,-2.196653,...,-2.131941,-1.488998,-0.062865,-0.122249,2.217655,1.203446,1.180406,0.273386,0.130200,2020-07
2,84.83,6.0,2013.0,7.401580,7.0,1.000000,0.274194,-0.137119,1.368243,-1.201620,...,-2.031952,-1.270553,-0.651409,-1.210025,1.739807,1.211842,1.265154,0.637330,-0.447079,2020-07
3,59.75,13.0,2016.0,7.066081,4.0,1.000000,0.402139,0.171405,0.707894,-0.902115,...,-1.890251,-0.957274,-0.138104,-0.749877,1.704004,1.395738,0.735970,0.548623,-0.146055,2020-07
4,49.94,7.0,1989.0,6.967225,31.0,0.000000,1.091872,0.706208,0.801780,-1.592087,...,-2.315390,-1.987213,0.255648,0.739055,2.142885,1.633339,1.013102,0.846412,0.591645,2020-07


In [25]:
merged_df = pd.merge(sale, monthly_sentiment, on='month', how='left')

In [26]:
merged_df.drop('month', axis=1, inplace=True)

In [27]:
merged_df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,sentiment_score
0,59.91,10.0,1998.0,6.766156,22.0,0.266667,1.840863,0.455314,1.392201,-1.213092,...,-2.512005,-1.100615,-0.027693,-1.204624,2.213226,1.975165,0.900340,0.369453,0.569277,0.014421
1,59.77,7.0,1996.0,7.499383,24.0,1.000000,0.512528,0.327763,0.756215,-2.196653,...,-2.131941,-1.488998,-0.062865,-0.122249,2.217655,1.203446,1.180406,0.273386,0.130200,0.014421
2,84.83,6.0,2013.0,7.401580,7.0,1.000000,0.274194,-0.137119,1.368243,-1.201620,...,-2.031952,-1.270553,-0.651409,-1.210025,1.739807,1.211842,1.265154,0.637330,-0.447079,0.014421
3,59.75,13.0,2016.0,7.066081,4.0,1.000000,0.402139,0.171405,0.707894,-0.902115,...,-1.890251,-0.957274,-0.138104,-0.749877,1.704004,1.395738,0.735970,0.548623,-0.146055,0.014421
4,49.94,7.0,1989.0,6.967225,31.0,0.000000,1.091872,0.706208,0.801780,-1.592087,...,-2.315390,-1.987213,0.255648,0.739055,2.142885,1.633339,1.013102,0.846412,0.591645,0.014421


In [28]:
len(merged_df)

23217

In [29]:
merged_df.to_csv("../data/final/90_senti.csv",index=False)